%md
# Healthcare Patient Deterioration Analytics
**Student:** Vansh Jain \
**Technologies Used:** Azur Data Factory · Databricks PySpark · Unity Catalog · Spark MLlib \
**Notebook:** Machine Learning Layer\
**Layers:** Bronze → Silver → Gold

### Importing Spark MLlib and other relevant libraries

In [0]:
from pyspark.ml.feature import (
    StringIndexer, 
    OneHotEncoder, 
    VectorAssembler
)

### Configuring Storage Account

In [0]:
storage_account = 'healthcaremedalliondev'
storage_key = '<Your Storage Account Key>'

In [0]:
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

In [0]:
silver_path = f'abfss://silver@{storage_account}.dfs.core.windows.net/silver_patient_vitals'

### Reading Data From Silver Delta Table

In [0]:
silver_df = (
    spark.read
    .format('delta')
    .load(silver_path)
)

### Defining Relevant Columns

In [0]:
ml_df = silver_df.select(
    'heart_rate',
    'respiratory_rate',
    'spo2_pct',
    'temperature_c',
    'systolic_bp',
    'diastolic_bp',
    'pulse_pressure',
    'oxygen_flow',
    'mobility_score',
    'wbc_count',
    'lactate',
    'creatinine',
    'crp_level',
    'hemoglobin',
    'sepsis_risk_score',
    'age',
    'comorbidity_index',
    'is_fever',
    'is_tachycardia',
    'is_hypoxic',
    'high_sepsis_risk',
    'gender',
    'admission_type',
    'oxygen_device',
    'deterioration_next_12h'
)

### Basic Data Validation

In [0]:
print(ml_df.count())
ml_df.printSchema()
display(ml_df.limit(5))

### Categorical COlumns

In [0]:
categorical_cols = [
    'gender',
    'admission_type',
    'oxygen_device'
]

### String Indexing

Convert categorical values into numeric indices.

In [0]:
indexers =  [
    StringIndexer(
        inputCol = col,
        outputCol=f'{col}_index',
        handleInvalid='keep'
    )
    for col in categorical_cols
]

### One-Hot Encoding
Spark should not interpret category indices as ordinal values

In [0]:
encoder = OneHotEncoder(
    inputCols = [f'{c}_index' for c in categorical_cols],
    outputCols = [f'{c}_vec' for c in categorical_cols]
)

### Feature List

Complete List of Model Related Feature Columns

In [0]:
feature_columns = [
    "heart_rate",
    "respiratory_rate",
    "spo2_pct",
    "temperature_c",
    "systolic_bp",
    "diastolic_bp",
    "pulse_pressure",
    "oxygen_flow",
    "mobility_score",
    "wbc_count",
    "lactate",
    "creatinine",
    "crp_level",
    "hemoglobin",
    "sepsis_risk_score",
    "age",
    "comorbidity_index",
    "is_fever",
    "is_tachycardia",
    "is_hypoxic",
    "high_sepsis_risk",
    "gender_vec",
    "admission_type_vec",
    "oxygen_device_vec"
]

### Vector Assembler

All features in one vector.

In [0]:
assembler = VectorAssembler(
    inputCols = feature_columns,
    outputCol = 'features'
)

### Label

Naming Target column as label

In [0]:
from pyspark.sql.functions import col

ml_df = ml_df.withColumn(
    'label',
    col('deterioration_next_12h').cast('double')
)

### Create Pipeline

In [0]:
from pyspark.ml import Pipeline

pipeline = Pipeline(
    stages = [
        *indexers,
        encoder,
        assembler
    ]
)

### Train - Test Split
Split the dataset into:

80% Training Data
20% Testing Data

In [0]:
train_df, test_df = ml_df.randomSplit(
    [0.8, 0.2],
    seed = 42
)

In [0]:
train_df.groupby('deterioration_next_12h').count().show()

In [0]:
test_df.groupby('deterioration_next_12h').count().show()

### Fit the Pipeline

In [0]:
pipeline_model = pipeline.fit(train_df)

### Transform the Dataset

In [0]:
train_processed = pipeline_model.transform(train_df)
test_processed = pipeline_model.transform(test_df)

### Verify Features and Labels

In [0]:
train_processed.select(
    'features',
    'label'
).show(5, truncate = False)

In [0]:
test_processed.select(
    'features',
    'label'
).show(5, truncate= False)

### Verify Schema

In [0]:
train_processed.printSchema()

### Creating Simplified DataFrames
Simplify the training and testing DataFrames

In [0]:
train_final = train_processed.select(
    'features',
    'label'
)

test_final = test_processed.select(
    'features',
    'label'
)

In [0]:
train_final.printSchema()
test_final.printSchema()

# Model Training

In [0]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

## Model Initializing

In [0]:
rf = RandomForestClassifier(
    featuresCol = 'features',
    labelCol = 'label',
    predictionCol = 'prediction',
    probabilityCol = 'probability',
    rawPredictionCol = 'rawPrediction',
    numTrees = 100,
    maxDepth = 10,
    seed = 42
)

## Training the model

In [0]:
rf_model = rf.fit(train_final)

## Prediction generation and validation

In [0]:
predictions = rf_model.transform(test_final)

In [0]:
predictions.select(
    'label',
    'prediction',
    'probability'
).show(10, truncate=False)

## Model evaluation

In [0]:
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol='label',
    predictionCol='prediction',
    metricName='accuracy'
)
accuracy = accuracy_evaluator.evaluate(predictions)
print(f'Accuracy : {accuracy : .4f}')

In [0]:
precision_evaluator = MulticlassClassificationEvaluator(
    labelCol='label',
    predictionCol='prediction',
    metricName='weightedPrecision'
)
precision = precision_evaluator.evaluate(predictions)
print(f'Precision : {precision: .4f}')

In [0]:
recall_evaluator = MulticlassClassificationEvaluator(
    labelCol='label',
    predictionCol='prediction',
    metricName='weightedRecall'
)
recall = recall_evaluator.evaluate(predictions)
print(f'Recall : {recall : .4f}')

In [0]:
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol='label',
    predictionCol='prediction',
    metricName='f1'
)
f1 = f1_evaluator.evaluate(predictions)
print(f'F1 score : {f1 : .4f}')

In [0]:
auc_evaluator = BinaryClassificationEvaluator(
    labelCol='label',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderROC'
)
auc = auc_evaluator.evaluate(predictions)
print(f'ROC-AUC: {auc: .4f}')

In [0]:
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

## Confusion Matrix

In [0]:
predictions.groupBy(
    'label',
    'prediction'
).count().orderBy('label', 'prediction').show()

## Feature Importance

In [0]:
feature_importance = rf_model.featureImportances

In [0]:
importance_list = list(zip(feature_columns, feature_importance.toArray()))

In [0]:
importance_df = spark.createDataFrame(
    [(f, float(i)) for f, i in importance_list],
    ['Feature', 'Importance']
)

In [0]:
from pyspark.sql.functions import desc

importance_df = (
    importance_df
    .orderBy(desc('Importance'))
)

In [0]:
display(importance_df)

In [0]:
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import udf

get_deterioration_probability = udf(
    lambda v: float(v[1]),
    DoubleType()
)

prediction_df = (
    predictions
    .withColumn(
        "deterioration_probability",
        get_deterioration_probability(col("probability"))
    )
    .select(
        "label",
        "prediction",
        "deterioration_probability"
    )
)

In [0]:
prediction_path = (
    f'abfss://gold@{storage_account}.dfs.core.windows.net/'
    'gold_ml_predictions'
)

In [0]:
(
    predictions_df.write
    .format('delta')
    .mode('overwrite')
    .option("overwriteSchema", "true")
    .save(prediction_path)
)

In [0]:
prediction_table = (
    spark.read
    .format('delta')
    .load(prediction_path)
)
display(prediction_table.limit(10))

## Saving Trained Model

In [0]:
model_path = f'abfss://gold@{storage_account}.dfs.core.windows.net/model/random_forest'
rf_model.write().overwrite().save(model_path)

In [0]:
from pyspark.ml.classification import RandomForestClassificationModel
laoded_model = RandomForestClassificationModel.load(model_path)